# Experiment 09 — Structural memory: attachment loss and the limits of plaque control

**Idea.** Response-state memory ($\theta$) is reversible: it decays on the memory timescale $1/\varepsilon$. Periodontitis also leaves a memory that does not decay, namely attachment loss. We add one slow, irreversible variable $D\in[0,1]$:

$$\dot D=\delta\,[H-h_D]_+\,(1-D),\qquad \dot M=r\,(-M+G(H)+\gamma D).$$

Tissue is lost only during established inflammation ($H>h_D=0.8$), so cyclic gingivitis does not cause attachment loss. Loss saturates at $D=1$. The pocket left behind acts as a niche that supports the subgingival community ($\gamma D$) independently of the host response.

The system stays cooperative: $D$ increases with $H$, and $M$ increases with $D$.

**Lemma (exact).** At any fixed $D$, the equilibria are those of M1 with forcing $u+\gamma D$. The pocket acts as an endogenous plaque reservoir that home care cannot reach, and the whole bifurcation diagram slides left by $\gamma D$. Two critical attachment levels follow.
* $D_{\text{irr}}=u_\downarrow/\gamma$: beyond it, perfect plaque control ($u=0$) no longer guarantees a return to health.
* $D_{\text{spont}}=u_\uparrow/\gamma$: beyond it, no healthy state exists even with $u=0$.

These define three stages, which map onto the logic of stepwise periodontal therapy (supragingival plaque control, then subgingival instrumentation, then surgical correction of residual pockets).

**Hosts.**
* Slow memory, $\varepsilon\in\{0.1,0.03,0.01\}$.
* Instant adaptation (same equilibria, no memory timescale).

Subgingival recolonisation rate $r\in\{1,0.3,0.1,0.03,0.01\}$.

**Sections**
* **A** stage structure (analytic, with a numerical check of the lemma)
* **B** natural history under maintenance (Exp 07 protocol, with $D$ accumulating)
* **C** the treatment ladder: hygiene, debridement, repeated debridement, structural correction
* **D** maintenance after therapy: how residual attachment loss and residual memory set the tolerable plaque level
* **E** checks and summary

In [ ]:
# =====================================================================
# EXPERIMENT 09 — folder setup
# =====================================================================
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"
EXP_DIR = BASE_DIR / "Experiment_09_Structural_Memory"
EXP_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("EXP_QUICK") == "1"     # set True for a fast smoke test with reduced grids
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}   QUICK={QUICK}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq

versions_text = f"""
Experiment: 09 Structural memory (attachment loss)
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

# ---- figure style (same tokens as Experiment 07) ------------------------
INK, INK2, GRIDC, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# CORE MODEL — state (H, M, A, D), A = ln(theta0/theta), D = structural memory
# =====================================================================
#   dH/dt = -H + F(M + u(t); theta0*exp(-A))
#   dM/dt =  r*(-M + G(H) + gamma*D)              pocket niche supports the community
#   dA/dt =  eps*(alpha*H - A)                     response-state memory ('adaptive')
#   dD/dt =  delta*max(H - hD, 0)*(1 - D)          irreversible, saturating tissue loss
# mode 'adaptive' (slow memory) | 'instant' (A = alpha*H: same equilibria, no memory
# timescale) | 'frozen' (A held fixed).  Fixed-step RK4, vectorised over elements.
COMPARATOR = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)   # Exp 06B backbone
DT = 0.1
U_UP, U_DOWN = 0.4682392215, 0.0630204768                       # static folds (D = 0)


def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def rhs(H, M, A, D, u, eps, p, mode, gamma, delta, hD):
    A_eff = p['a'] * H if mode == 'instant' else A
    dH = -H + hill(M + u, p['th0'] * np.exp(-A_eff), p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']) + gamma * D)
    dA = eps * (p['a'] * H - A) if mode == 'adaptive' else 0.0 * A
    dD = delta * np.maximum(H - hD, 0.0) * (1.0 - D)
    return dH, dM, dA, dD


def run(y0, u_fn, n_steps, eps=0.0, mode='adaptive', p=COMPARATOR, gamma=0.0,
        delta=0.0, hD=0.8, events=None, recorder=None, dt=DT):
    """u_fn(i, c): forcing for step i at stage offset c in {0, .5, 1};
    events(i, H, M, A, D) -> state, applied at the start of step i;
    recorder(i, H, M, A, D), called after each step."""
    H, M, A, D = [np.array(v, float, copy=True) for v in np.broadcast_arrays(*y0)]
    eps, gamma, delta = [np.broadcast_to(np.asarray(v, float), H.shape) for v in (eps, gamma, delta)]
    if isinstance(p.get('r'), np.ndarray):
        p = dict(p, r=np.broadcast_to(p['r'], H.shape))
    f = lambda h, m, a, d, u: rhs(h, m, a, d, u, eps, p, mode, gamma, delta, hD)
    for i in range(n_steps):
        if events is not None:
            H, M, A, D = events(i, H, M, A, D)
        u0, uh, u1 = u_fn(i, 0.0), u_fn(i, 0.5), u_fn(i, 1.0)
        k1 = f(H, M, A, D, u0)
        k2 = f(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], D + .5*dt*k1[3], uh)
        k3 = f(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], D + .5*dt*k2[3], uh)
        k4 = f(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], D + dt*k3[3], u1)
        H = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        D = D + dt/6 * (k1[3] + 2*k2[3] + 2*k3[3] + k4[3])
        if recorder is not None:
            recorder(i, H, M, A, D)
    if mode == 'instant':
        A = p['a'] * H
    return H, M, A, D


def piecewise_u(boundaries, values):
    """Piecewise-constant forcing: segment k+1 starts at step boundaries[k] (per element)."""
    B = np.stack([np.asarray(b, np.int64) for b in boundaries])
    V = np.stack([np.asarray(v, float) for v in values])
    def u_fn(i, c):
        seg = (i >= B).sum(axis=0)
        return np.take_along_axis(V, seg[None, :], 0)[0]
    return u_fn


def u_branch(H, p=COMPARATOR, gamma=0.0, D=0.0):
    """Equilibrium forcing as a function of H* (M1 with structural memory D frozen)."""
    th = p['th0'] * np.exp(-p['a'] * H)
    return th * (H / (1 - H)) ** (1 / p['n']) - hill(H, p['K'], p['q']) - gamma * D


def low_equilibrium(u, p=COMPARATOR, gamma=0.0, D=0.0):
    Hs = np.linspace(1e-9, 0.6, 20001)
    f = u_branch(Hs, p, gamma, D) - u
    i = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0][0]
    H = brentq(lambda h: u_branch(h, p, gamma, D) - u, Hs[i], Hs[i + 1], xtol=1e-15)
    return H, hill(H, p['K'], p['q']) + gamma * D, p['a'] * H


def high_equilibrium(u, p=COMPARATOR, gamma=0.0, D=0.0):
    Hs = 1 - np.logspace(-12, np.log10(0.5), 20001)
    f = u_branch(Hs, p, gamma, D) - u
    i = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0][0]
    H = brentq(lambda h: u_branch(h, p, gamma, D) - u, Hs[i + 1], Hs[i], xtol=1e-15)
    return H, hill(H, p['K'], p['q']) + gamma * D, p['a'] * H

In [ ]:
# =====================================================================
# PARAMETERS
# =====================================================================
GAMMA = 0.8           # niche support of the community per unit attachment loss
HD = 0.8              # host response above which tissue is lost
DELTAS = [0.0003, 0.001, 0.003]   # rates of attachment loss
D_IRR, D_SPONT = U_DOWN / GAMMA, U_UP / GAMMA
K_REGROW, T_MAINT = 0.05, 60.0    # Exp 07 maintenance protocol
U_PRE = 0.20          # plaque level of the untreated diseased site (pre-therapy)
R_LIST = [1.0, 0.3, 0.1, 0.03, 0.01]
HOSTS = [('instant', 0.0), ('adaptive', 0.1), ('adaptive', 0.03), ('adaptive', 0.01)]
D_GRID = np.round(np.arange(0.0, 0.8001, 0.01), 3)
THERAPIES = ['hygiene', 'debridement', 'repeated debridement', 'structural correction']
DEBRIDE_EVERY, DEBRIDE_SESSIONS, STRUCT_FACTOR = 20.0, 10, 0.25
NAT_CYCLES, NAT_PEAKS = 100, [0.55, 0.62]
D_RES = np.round(np.arange(0.0, 0.301, 0.05), 3)
PK = np.round(np.arange(0.20, 0.7001, 0.01), 3)
if QUICK:
    HOSTS = HOSTS[:2]; R_LIST = [1.0, 0.1]; D_GRID = np.round(np.arange(0, 0.801, 0.1), 3)
    DELTAS = [0.001]; NAT_CYCLES = 30; D_RES = np.array([0.0, 0.2]); PK = np.round(np.arange(0.2, 0.701, 0.05), 3)

host_name = lambda mode, eps: 'instant' if mode == 'instant' else f'eps={eps}'
with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=COMPARATOR, gamma=GAMMA, h_D=HD, deltas=DELTAS, D_irr=D_IRR, D_spont=D_SPONT,
                   k_regrow=K_REGROW, T_maint=T_MAINT, u_pre_therapy=U_PRE, r_list=R_LIST,
                   hosts=[host_name(*h) for h in HOSTS], therapies=THERAPIES,
                   debride_every=DEBRIDE_EVERY, debride_sessions=DEBRIDE_SESSIONS,
                   structural_factor=STRUCT_FACTOR, dt=DT), f, indent=4)
print(f"D_irr = {D_IRR:.4f}   D_spont = {D_SPONT:.4f}")

def periodic_u(umax):
    Ns = int(round(T_MAINT / DT)); umax = np.asarray(umax, float)
    return lambda i, c: umax * (1 - np.exp(-K_REGROW * (((i % Ns) + c) * DT)))
peak_to_umax = lambda pk: np.asarray(pk) / (1 - np.exp(-K_REGROW * T_MAINT))

## A. Stage structure

Left panel: the equilibrium branch $H^*(u)$ at several attachment levels; dashed segments are unstable. Right panel: fold lines in the $(D,u)$ plane, $u=u_\uparrow-\gamma D$ and $u=u_\downarrow-\gamma D$. The line $u=0$ (perfect plaque control) crosses them at $D_{\text{irr}}$ and $D_{\text{spont}}$.

The lemma is then checked numerically by long simulations from low and high initial states.

In [ ]:
Hs = 1 - np.logspace(-6, 0, 4000)[::-1] * 0.999999
Hs = np.sort(np.clip(Hs, 1e-6, 1 - 1e-6))
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.5), constrained_layout=True)
rows = []
for Dv, col in zip([0.0, D_IRR, 0.3, D_SPONT], ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]):
    u = u_branch(Hs, gamma=GAMMA, D=Dv)
    du = np.gradient(u, Hs)
    st = du > 0
    for seg_stable in [True, False]:
        y = np.where(st == seg_stable, Hs, np.nan)
        ax[0].plot(u, y, '-' if seg_stable else '--', color=col, lw=1.6 if seg_stable else 1.0,
                   label=f"D = {Dv:.3f}" if seg_stable else None)
    for uu, hh, s_ in zip(u[::20], Hs[::20], st[::20]):
        rows.append(dict(D=Dv, u=uu, H=hh, stable=s_))
ax[0].axvline(0, color=INK2, lw=0.9)
ax[0].set(xlim=(-0.6, 0.8), xlabel="plaque forcing u", ylabel="equilibrium host response H*",
          title="The diagram slides left as D accumulates")
ax[0].legend(fontsize=7, loc="center right")
Dl = np.linspace(0, 1, 200)
ax[1].plot(Dl, U_UP - GAMMA * Dl, color=INK, lw=1.4, label=r"upward fold $u_\uparrow-\gamma D$")
ax[1].plot(Dl, U_DOWN - GAMMA * Dl, color=INK, lw=1.4, ls='--', label=r"downward fold $u_\downarrow-\gamma D$")
ax[1].fill_between(Dl, U_DOWN - GAMMA * Dl, U_UP - GAMMA * Dl, color=GRIDC, alpha=0.7, lw=0, label="bistable")
ax[1].axhline(0, color=INK2, lw=0.9)
for xv, lab in [(D_IRR, r"$D_{irr}$"), (D_SPONT, r"$D_{spont}$")]:
    ax[1].axvline(xv, color=INK2, lw=0.8, ls=':'); ax[1].text(xv + 0.01, 0.47, lab, fontsize=8, color=INK2)
for x0, x1, txt in [(0, D_IRR, "1"), (D_IRR, D_SPONT, "2"), (D_SPONT, 1.0, "3")]:
    ax[1].text((x0 + x1) / 2, 0.63, txt, ha='center', fontsize=8, color=INK, fontweight='bold')
ax[1].text(0.995, 0.63, "stage", ha='right', fontsize=7, color=INK2, transform=ax[1].get_yaxis_transform())
ax[1].set(xlim=(0, 1), ylim=(-0.7, 0.7), xlabel="attachment loss D", ylabel="plaque forcing u",
          title="Stages at perfect plaque control (u = 0)")
ax[1].legend(fontsize=6.5, loc="lower left")
pd.DataFrame(rows).to_csv(EXP_DIR / "fig09_01_stage_structure_data.csv", index=False)
save_fig(fig, "fig09_01_stage_structure")

# numerical check of the lemma: long runs at frozen D from low and high initial states
chk = []
for Dv in [0.0, 0.1, 0.3, 0.5]:
    for u in [0.0, 0.1, 0.3]:
        Hl = Hh = np.nan
        Hf = run((np.array([0.0, 0.99]), np.array([0.0, 1.0]), np.array([0.0, 1.0]), Dv),
                 lambda i, c: np.full(2, u), 20000, eps=0.1, gamma=GAMMA)[0]
        if u + GAMMA * Dv <= 0: Hl = 0.0
        else:
            try: Hl = low_equilibrium(u, gamma=GAMMA, D=Dv)[0]
            except IndexError: pass
        try: Hh = high_equilibrium(u, gamma=GAMMA, D=Dv)[0]
        except IndexError: pass
        chk.append(dict(D=Dv, u=u, u_eff=u + GAMMA * Dv, sim_from_low=Hf[0], sim_from_high=Hf[1],
                        analytic_low=Hl, analytic_high=Hh))
lemma = pd.DataFrame(chk); lemma.to_csv(EXP_DIR / "lemma_check.csv", index=False)
display(lemma.round(5))

## B. Natural history under maintenance

This is the Exp 07 protocol ($T=60$, plaque regrowing to a peak before each reset), now with attachment loss accumulating. Peak 0.55 lies below the slow-memory ratchet threshold (about 0.570) and above the instant-host threshold (about 0.496). Peak 0.62 lies above both.

In [ ]:
Ns = int(round(T_MAINT / DT))
rows = []
for mode, eps in [h for h in HOSTS if h[0] == 'instant' or h[1] == 0.01] or HOSTS:
    pk = np.repeat(NAT_PEAKS, len(DELTAS)); dl = np.tile(DELTAS, len(NAT_PEAKS))
    cyc = {'D': [], 'Hmin': [], 'Hmax': [], 'theta': []}
    state = {'hmin': None, 'hmax': None}
    def rec(i, H, M, A, D):
        if i % Ns == 0:
            state['hmin'], state['hmax'] = H.copy(), H.copy()
            cyc['theta'].append(COMPARATOR['th0'] * np.exp(-(A if mode == 'adaptive' else COMPARATOR['a'] * H)))
        state['hmin'] = np.minimum(state['hmin'], H); state['hmax'] = np.maximum(state['hmax'], H)
        if (i + 1) % Ns == 0:
            cyc['D'].append(D.copy()); cyc['Hmin'].append(state['hmin'].copy()); cyc['Hmax'].append(state['hmax'].copy())
    run((np.zeros(len(pk)), 0.0, 0.0, 0.0), periodic_u(peak_to_umax(pk)), NAT_CYCLES * Ns, eps=eps, mode=mode,
        gamma=GAMMA, delta=dl, hD=HD, recorder=rec)
    for k in range(len(pk)):
        for c in range(NAT_CYCLES):
            rows.append(dict(host=host_name(mode, eps), peak=pk[k], delta=dl[k], cycle=c, D=cyc['D'][c][k],
                             H_min=cyc['Hmin'][c][k], H_max=cyc['Hmax'][c][k], theta_at_reset=cyc['theta'][c][k]))
dfB = pd.DataFrame(rows); dfB.to_csv(EXP_DIR / "fig09_02_natural_history_data.csv", index=False)

def first_cycle(series, cond):
    idx = np.where(cond(series))[0]
    return idx[0] if idx.size else np.nan
ms = []
for (h, pk, dl), g in dfB.groupby(['host', 'peak', 'delta']):
    g = g.sort_values('cycle')
    ent = g.H_min.values > 0.5
    ms.append(dict(host=h, peak=pk, delta=dl, cycle_entrapped=first_cycle(ent, lambda x: x),
                   cycle_D_irr=first_cycle(g.D.values, lambda x: x > D_IRR),
                   cycle_D_spont=first_cycle(g.D.values, lambda x: x > D_SPONT), D_final=g.D.values[-1]))
dfBm = pd.DataFrame(ms); dfBm.to_csv(EXP_DIR / "natural_history_milestones.csv", index=False)
display(dfBm)

fig, ax = plt.subplots(1, len(NAT_PEAKS), figsize=(9.5, 3.4), constrained_layout=True, sharey=True)
ax = np.atleast_1d(ax)
for a_, pk in zip(ax, NAT_PEAKS):
    for (h, dl), g in dfB[dfB.peak == pk].groupby(['host', 'delta']):
        col = INSTANT_COL if h == 'instant' else MEM_COL[0.01]
        ls = {0.0003: ':', 0.001: '-', 0.003: '--'}.get(dl, '-')
        a_.plot(g.cycle, g.D, ls, color=col, label=f"{h}, " + rf"$\delta$={dl}")
    for yv, lab in [(D_IRR, r"$D_{irr}$"), (D_SPONT, r"$D_{spont}$")]:
        a_.axhline(yv, color=INK2, lw=0.8, ls=':'); a_.text(1, yv + 0.015, lab, fontsize=7.5, color=INK2)
    a_.set(xlabel="maintenance cycle", title=f"peak plaque {pk} ({pk / U_UP:.2f} x static fold)")
ax[0].set_ylabel("attachment loss D"); ax[-1].legend(fontsize=6.5, loc="center right")
save_fig(fig, "fig09_02_natural_history")

## C. The treatment ladder

Diseased sites are taken at the high equilibrium for pre-therapy plaque $u=0.20$, at each attachment level $D$. Memory takes its high-state value ($\theta\approx0.30$). At $t=0$ one of four therapies is applied, always followed by perfect plaque control ($u=0$):
1. **hygiene** only;
2. **debridement**: $M\to0$ once;
3. **repeated debridement**: $M\to0$ every 20, ten sessions;
4. **structural correction**: $D\to0.25D$, plus debridement.

$D$ is frozen during follow-up, so outcomes are evaluated at a fixed attachment level. Follow-up lasts $\max(20/\varepsilon, 2000)$. A site is *resolved* if it ends on the low branch ($H<0.5$). Time to resolution is the last time at which $H\ge0.5$.

In [ ]:
def ladder(mode, eps):
    DD, RR, TH = np.meshgrid(D_GRID, R_LIST, np.arange(len(THERAPIES)), indexing='ij')
    DD, RR, TH = DD.ravel(), RR.ravel(), TH.ravel()
    n = DD.size
    H0 = np.array([high_equilibrium(U_PRE, gamma=GAMMA, D=d)[0] for d in D_GRID])
    idx = np.searchsorted(D_GRID, DD)
    H = H0[idx]; M = hill(H, COMPARATOR['K'], COMPARATOR['q']) + GAMMA * DD; A = COMPARATOR['a'] * H
    D = np.where(TH == 3, STRUCT_FACTOR * DD, DD)
    M = np.where(TH >= 1, 0.0, M)
    rep = TH == 2; every = int(round(DEBRIDE_EVERY / DT))
    def ev(i, H, M, A, Dv):
        if i > 0 and i % every == 0 and i <= every * (DEBRIDE_SESSIONS - 1):
            M = np.where(rep, 0.0, M)
        return H, M, A, Dv
    last_high = np.full(n, -1.0)
    def rec(i, H, M, A, Dv):
        last_high[H >= 0.5] = (i + 1) * DT
    follow = max(20 / eps, 2000.0) if mode == 'adaptive' else 2000.0
    Hf = run((H, M, A, D), lambda i, c: np.zeros(n), int(round(follow / DT)), eps=eps, mode=mode,
             p=dict(COMPARATOR, r=RR), gamma=GAMMA, delta=0.0, events=ev, recorder=rec)[0]
    return pd.DataFrame(dict(host=host_name(mode, eps), D=DD, r=RR, therapy=[THERAPIES[t] for t in TH],
                             resolved=Hf < 0.5, H_final=Hf,
                             time_to_resolution=np.where(Hf < 0.5, np.maximum(last_high, 0.0), np.nan)))
parts = []
for mode, eps in HOSTS:
    t0 = time.time(); parts.append(ladder(mode, eps)); print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfC = pd.concat(parts, ignore_index=True)
dfC.to_csv(EXP_DIR / "fig09_03_treatment_ladder_data.csv", index=False)

def d_max(g):
    """largest D such that every D up to it is resolved"""
    g = g.sort_values('D'); ok = g.resolved.values
    return g.D.values[np.argmin(ok) - 1] if (~ok).any() and np.argmin(ok) > 0 else (g.D.values[-1] if ok.all() else np.nan)
dmax = dfC.groupby(['host', 'r', 'therapy']).apply(d_max, include_groups=False).rename('D_max_resolved').reset_index()
dmax.to_csv(EXP_DIR / "treatment_ladder_Dmax.csv", index=False)
display(dmax.pivot_table(index=['therapy', 'r'], columns='host', values='D_max_resolved').round(3))

In [ ]:
hosts_show = [h for h in ['instant', 'eps=0.01'] if h in dfC.host.unique()] or [dfC.host.unique()[0]]
r_show = [r for r in [1.0, 0.1] if r in R_LIST]
fig, axs = plt.subplots(len(r_show), len(hosts_show), figsize=(4.8 * len(hosts_show), 1.7 * len(r_show) + 0.9),
                        constrained_layout=True, squeeze=False)
for i_r, r in enumerate(r_show):
    for j_h, h in enumerate(hosts_show):
        a_ = axs[i_r, j_h]
        for k, th in enumerate(THERAPIES):
            g = dfC[(dfC.host == h) & (dfC.r == r) & (dfC.therapy == th)].sort_values('D')
            a_.scatter(g.D, np.full(len(g), k), s=14, marker='s',
                       c=np.where(g.resolved, "#1c5cab", "#f0efec"), edgecolors=np.where(g.resolved, "#1c5cab", "#c3c2b7"), lw=0.5)
        for xv in (D_IRR, D_SPONT):
            a_.axvline(xv, color=INK2, lw=0.8, ls=':')
        a_.set_yticks(range(len(THERAPIES)), THERAPIES if j_h == 0 else [''] * len(THERAPIES), fontsize=7)
        a_.set(xlim=(-0.02, D_GRID.max() + 0.02), ylim=(-0.6, len(THERAPIES) - 0.4),
               title=f"{h}, recolonisation r = {r}"); a_.grid(False)
        if i_r == len(r_show) - 1:
            a_.set_xlabel(r"attachment loss D   (dotted: $D_{irr}$, $D_{spont}$;  filled = resolved)")
save_fig(fig, "fig09_03_treatment_ladder")

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4), constrained_layout=True)
g1 = dmax[dmax.therapy == 'debridement']
for h, g in g1.groupby('host'):
    col = INSTANT_COL if h == 'instant' else MEM_COL[float(h.split('=')[1])]
    g = g.sort_values('r')
    ax[0].plot(g.r, g.D_max_resolved, 'o-', ms=3.5, color=col, label=h)
    if h != 'instant':
        e = float(h.split('=')[1]); ax[1].plot(g.r / e, g.D_max_resolved, 'o-', ms=3.5, color=col, label=h)
gi = g1[g1.host == 'instant']
if len(gi):
    ax[1].axhspan(gi.D_max_resolved.min(), gi.D_max_resolved.max() + 1e-3, color=INSTANT_COL, alpha=0.15, lw=0,
                  label="instant host (all r)")
for a_ in ax:
    a_.axhline(D_IRR, color=INK2, lw=0.8, ls=':'); a_.axhline(D_SPONT, color=INK2, lw=0.8, ls=':')
    a_.set(xscale='log', ylabel="largest D resolved by one debridement")
ax[0].set(xlabel="subgingival recolonisation rate r", title="Debridement reach depends on recolonisation")
ax[1].set(xlabel=r"recolonisation rate / memory rate  $r/\varepsilon$", title="Slow-memory hosts on r / eps")
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
g1.to_csv(EXP_DIR / "fig09_04_debridement_reach_data.csv", index=False)
save_fig(fig, "fig09_04_debridement_reach")

tr = dfC[(dfC.therapy == 'hygiene') & dfC.resolved & (dfC.D < D_IRR)].groupby('host').time_to_resolution.median()
print("median time to resolution under hygiene alone (stage 1):"); display(tr)

## D. Maintenance after therapy

Treated sites, with residual attachment loss $D_{\text{res}}$ (frozen) and initial memory $A_0$, return to the Exp 07 maintenance regime ($T=60$). $A_0=0$ means that memory has fully recovered; $A_0=A_{\text{high}}$ means that maintenance starts immediately after therapy. The output is the highest peak plaque the site tolerates indefinitely, and how quickly it reconverts above that level.

In [ ]:
A_HIGH = high_equilibrium(U_PRE)[2]
A0_LIST = [0.0, 0.5 * A_HIGH, A_HIGH]
rows = []
for mode, eps in [h for h in HOSTS if h[0] == 'instant' or h[1] == 0.01] or HOSTS:
    t0 = time.time()
    A0s = A0_LIST if mode == 'adaptive' else [0.0]
    DD, AA, PP = np.meshgrid(D_RES, A0s, PK, indexing='ij'); DD, AA, PP = DD.ravel(), AA.ravel(), PP.ravel()
    horizon = max(30 * T_MAINT, 40 / eps) if mode == 'adaptive' else 30 * T_MAINT
    nc = int(horizon / T_MAINT); Hmin = np.full((DD.size, nc), np.nan); cur = {}
    def rec(i, H, M, A, D):
        if i % Ns == 0: cur['m'] = H.copy()
        cur['m'] = np.minimum(cur['m'], H)
        if (i + 1) % Ns == 0: Hmin[:, (i + 1) // Ns - 1] = cur['m']
    run((np.zeros(DD.size), 0.0, AA, DD), periodic_u(peak_to_umax(PP)), nc * Ns, eps=eps, mode=mode,
        gamma=GAMMA, delta=0.0, recorder=rec)
    ent = Hmin > 0.5
    for k in range(DD.size):
        conv = bool(ent[k, -3:].all())
        first = np.where(~ent[k])[0]
        rows.append(dict(host=host_name(mode, eps), D_res=DD[k], A0=AA[k], peak=PP[k], converts=conv,
                         cycles_to_conversion=(first[-1] + 1 if first.size else 0) if conv else np.nan))
    print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfD = pd.DataFrame(rows); dfD.to_csv(EXP_DIR / "fig09_05_post_therapy_maintenance_data.csv", index=False)
thr = dfD[dfD.converts].groupby(['host', 'D_res', 'A0']).peak.min().rename('threshold_peak').reset_index()
thr.to_csv(EXP_DIR / "post_therapy_thresholds.csv", index=False)
display(thr.pivot_table(index='D_res', columns=['host', 'A0'], values='threshold_peak').round(3))

fig, ax = plt.subplots(figsize=(5, 3.4), constrained_layout=True)
for (h, a0), g in thr.groupby(['host', 'A0']):
    if h == 'instant':
        ax.plot(g.D_res, g.threshold_peak / U_UP, 'o-', ms=3.5, color=INSTANT_COL, label="instant adaptation")
    else:
        ls = {0.0: '-', 0.5: '--', 1.0: ':'}[round(a0 / A_HIGH, 1)]
        ax.plot(g.D_res, g.threshold_peak / U_UP, 'o' + ls, ms=3.5, color=MEM_COL[0.01],
                label=rf"$\varepsilon$=0.01, memory at start {a0 / A_HIGH:.0%} of diseased")
ax.axhline(1.0, color=INK2, ls='--', lw=0.9)
ax.set(xlabel="residual attachment loss after therapy", ylabel="tolerable peak plaque / static fold",
       title="What maintenance can tolerate after therapy")
ax.legend(fontsize=6.5)
save_fig(fig, "fig09_05_post_therapy_maintenance")

## E. Consistency checks and summary

1. The lemma (above).
2. Resolution should be downward closed in $D$ for every therapy, host and $r$.
3. Slower recolonisation should never reduce the reach of a single debridement.
4. The post-therapy threshold should not increase with residual $D$ or with initial memory.

In [ ]:
checks = []
lem_err = np.nanmax(np.abs(np.r_[lemma.sim_from_low - lemma.analytic_low.where(~lemma.analytic_low.isna(), lemma.sim_from_low),
                                 lemma.sim_from_high - lemma.analytic_high.where(~lemma.analytic_high.isna(), lemma.sim_from_high)]))
checks.append(dict(check="lemma: max |simulated - analytic equilibrium H|", value=lem_err))
dc = all(np.all(np.diff(g.sort_values('D').resolved.values.astype(int)) <= 0)
         for _, g in dfC.groupby(['host', 'r', 'therapy']))
checks.append(dict(check="resolution downward closed in D", value=dc))
mono_r = all(np.all(np.diff(g.sort_values('r', ascending=False).D_max_resolved.values) >= -1e-9)
             for _, g in dmax[dmax.therapy == 'debridement'].groupby('host'))
checks.append(dict(check="debridement reach non-decreasing as recolonisation slows", value=mono_r))
mono_t = all(np.all(np.diff(g.sort_values('D_res').threshold_peak.values) <= 1e-9) for _, g in thr.groupby(['host', 'A0']))
checks.append(dict(check="post-therapy threshold non-increasing in residual D", value=mono_t))
dfE = pd.DataFrame(checks); dfE.to_csv(EXP_DIR / "consistency_checks.csv", index=False); display(dfE)

summary = pd.concat([
    pd.DataFrame([dict(item="D_irr (analytic)", value=D_IRR), dict(item="D_spont (analytic)", value=D_SPONT)]),
    dmax.assign(item=lambda d: "D_max: " + d.therapy + ", " + d.host + ", r=" + d.r.astype(str))[['item', 'D_max_resolved']]
        .rename(columns={'D_max_resolved': 'value'}),
    thr.assign(item=lambda d: "post-therapy threshold: " + d.host + ", D_res=" + d.D_res.astype(str) +
               ", A0=" + d.A0.round(3).astype(str))[['item', 'threshold_peak']].rename(columns={'threshold_peak': 'value'}),
], ignore_index=True)
summary.to_csv(EXP_DIR / "summary.csv", index=False)
print(len(summary), "summary rows saved")

In [ ]:
readme = """Experiment 09 - Structural memory (attachment loss)
===================================================
M1 (comparator backbone) plus D: dD/dt = delta*[H - 0.8]_+*(1 - D);
dM/dt = r(-M + G(H) + gamma*D), gamma = 0.8. Lemma: at fixed D the equilibria are those
of M1 with forcing u + gamma*D, giving D_irr = u_down/gamma and D_spont = u_up/gamma.
Hosts: slow memory eps = 0.1, 0.03, 0.01; instant adaptation (same equilibria).

Files
- parameters.json, software_versions.txt
- fig09_01_stage_structure, lemma_check.csv           equilibrium branches vs D; stage map; lemma check
- fig09_02_natural_history, natural_history_milestones.csv   D accumulation under maintenance (T = 60)
- fig09_03_treatment_ladder(_data.csv), treatment_ladder_Dmax.csv   four therapies x D x r x host
- fig09_04_debridement_reach                           largest D resolved by one debridement vs r and r/eps
- fig09_05_post_therapy_maintenance, post_therapy_thresholds.csv   tolerable peak plaque vs residual D and memory
- consistency_checks.csv, summary.csv
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_09_Structural_Memory_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)